# Quantum Physics-Informed Neural Networks (QPINNs)
## Solving PDEs and ODEs with Variational Quantum Circuits

**Morten Hjorth-Jensen**, Department of Physics & CCSE, University of Oslo

*Synchronised with `week15_merged.tex` — May 2026*

---

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | QNN recap: parameter-shift, QFI, TDVP, barren plateaus |
| 3 | Input encoding: angle, IQP, data re-uploading, Fourier view |
| 4 | Classical PINNs |
| 5 | QPINN foundation: unified shift framework |
| 6 | 1D Poisson: strong form + Ritz |
| 7 | Harmonic oscillator ODE |
| 8 | Heat equation (space-time encoding) |
| 9 | Quantum Fisher Information and natural gradient |
| 10 | Expressivity: Fourier analysis and data re-uploading |
| 11 | Exercises |
| 12 | Summary |


## 1  Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
print("Imports OK.")


---
## 2  Recap: Quantum Neural Networks (week15_merged.tex, Section 0)

### 2.1  Variational quantum state

$$|\psi(x,\theta)\rangle = U(\theta)\,U(x)\,|0\rangle^{\otimes n}, \qquad f(x,\theta) = \langle\psi|\,O\,|\psi\rangle$$

- $U(x)$: feature map (encodes input)  
- $U(\theta)$: variational ansatz (trainable)  
- $O = Z_0$: observable

### 2.2  Parameter-shift rule

For $G(\theta) = e^{-i\theta P/2}$ (Pauli $P$, eigenvalues $\pm 1/2$):

$$\frac{\partial f}{\partial\theta} = \frac{1}{2}\bigl[f(\theta+\tfrac{\pi}{2}) - f(\theta-\tfrac{\pi}{2})\bigr]$$

Exact; 2 circuit evaluations per parameter.

### 2.3  QFI and natural gradient

$$F_{jk} = 4\,\mathrm{Re}\!\left(\langle\partial_j\psi|\partial_k\psi\rangle - \langle\partial_j\psi|\psi\rangle\langle\psi|\partial_k\psi\rangle\right)$$

**Natural gradient** (Stokes et al. 2020): $\dot{\theta} = -F^{-1}\nabla_\theta\mathcal{L}$

**TDVP** (imaginary-time): $\sum_k F_{jk}\dot{\theta}_k = \partial E/\partial\theta_j$ — equals natural gradient VQE.

### 2.4  Barren plateaus

$\mathrm{Var}(\nabla_\theta\mathcal{L}) \sim 2^{-n}$ for deep random circuits.
Mitigation: shallow, physics-inspired ansätze.


---
## 3  Input Encoding (week15_merged.tex, Section 1)

The **feature map** $\Phi: \mathbb{R}^d \to (\mathbb{C}^2)^{\otimes n}$ determines
expressivity, the induced kernel, and — for QPINNs — the accessible spatial frequency spectrum.

### 3.1  Angle encoding

$$S(x) = R_y(\pi x)^{\otimes n}, \qquad R_y(\alpha) = e^{-i\alpha Y/2}$$

Each qubit has generator $Y/2$ with eigenvalues $\pm 1/2$.
The input-shift rule with **SHIFT = $\pi/2$ applied to $x$** gives exact derivatives.

**Key formulas** (week15_merged.tex Section 6, code convention):

$$\frac{\partial u_\theta}{\partial x} = \frac{1}{2}\bigl[u_\theta(x+\tfrac{\pi}{2}) - u_\theta(x-\tfrac{\pi}{2})\bigr]$$

$$\frac{\partial^2 u_\theta}{\partial x^2} = u_\theta(x+\tfrac{\pi}{2}) - 2u_\theta(x) + u_\theta(x-\tfrac{\pi}{2})$$

3 circuit evaluations for the Laplacian — no ancilla qubits.

### 3.2  Fourier theorem (Schuld et al. 2021)

Any QNN output is a partial Fourier series:
$$u_\theta(x) = \sum_{\omega\in\Omega} c_\omega(\theta)\,e^{i\omega x}$$
Frequency set $\Omega$ fixed by encoding gates; coefficients tuned by $\theta$.

### 3.3  Data re-uploading

Interleave encoding and variational layers $L$ times:
$U(x,\theta) = \prod_{\ell=1}^L[W_\ell(\theta)\,S(x)]$

Maximum frequency: $\omega_{\max} = Ln/2$.  Design rule: $Ln \geq 2\omega^\star$ for target PDE frequency $\omega^\star$.

### 3.4  Encoding comparison for PDEs

| Encoding | Qubits | $\omega_{\max}$ |
|---|---|---|
| Angle (L=1) | $n$ | $n/2$ |
| Re-uploading | $n$ | $Ln/2$ |
| IQP / ZZ | $n$ | $n^2/4$ |
| Amplitude | $n$ | $2^n/2$ |


### 3.4  Gate definitions and QPINN circuit

In [ ]:
# =====================================================================
#  Gate primitives -- 2-qubit QPINN circuit
#  Architecture:
#    Encoding:    Ry(pi*x) x Ry(pi*x)   [angle encoding, n=2 qubits]
#    Var. layer:  CNOT(0->1) @ Ry(p0) x Ry(p1)
#    Observable:  Z x I  (Z on qubit 0)
#
#  Consistent with week15_merged.tex Section 7 Python code.
#  Uses ONLY Ry + CNOT -- no Rz gates.
# =====================================================================

def ry(t):
    c, s = np.cos(t / 2), np.sin(t / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)

I2   = np.eye(2, dtype=complex)
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
Z0   = np.kron(np.diag([1., -1.]).astype(complex), I2)   # Z x I

def var_layer(p):
    """Variational layer: CNOT(0->1) @ Ry(p[0]) x Ry(p[1]).  4 params per layer."""
    return CNOT @ np.kron(ry(p[0]), ry(p[1]))

def circuit(x, params):
    """
    Two-qubit QPINN circuit.

    Encoding : Ry(pi*x) x Ry(pi*x)
    Ansatz   : 2 variational layers  (8 parameters total)
    Output   : <Z_0>  in [-1, 1]

    Input-shift convention (week15_merged.tex):
      SHIFT = pi/2 applied directly to x.
      Gate becomes Ry(pi*(x + pi/2)).
      This is exact because the gate generator has eigenvalues +/- 1/2.
    """
    state   = np.zeros(4, dtype=complex); state[0] = 1.0
    enc     = np.kron(ry(np.pi * x), ry(np.pi * x))
    U       = var_layer(params[4:8]) @ var_layer(params[:4]) @ enc
    psi     = U @ state
    return float((psi.conj() @ Z0 @ psi).real)

def circuit_batch(xs, params):
    """Evaluate circuit at all x in array xs."""
    return np.array([circuit(x, params) for x in xs])

def circuit_dirichlet(x, params):
    """
    Hard Dirichlet BC enforcement: u_hat(x) = u(x) - [(1-x)*u(0) + x*u(1)].
    The linear correction has zero second derivative, so u_hat''= u'' exactly.
    Boundary values are exactly zero.
    """
    u0 = circuit(0., params)
    u1 = circuit(1., params)
    return circuit(x, params) - (1 - x) * u0 - x * u1

# ── Input-shift rule: EXACT spatial derivatives ────────────────────────────
# Convention: SHIFT = pi/2 applied to x  (week15_merged.tex Section 6).
SHIFT = np.pi / 2

def du_dx(x, params):
    """First spatial derivative via input-shift rule (exact, no FD)."""
    return 0.5 * (circuit(x + SHIFT, params) - circuit(x - SHIFT, params))

def d2u_dx2(x, params):
    """Second spatial derivative (Laplacian) via input-shift rule (exact, 3 circuits)."""
    return (circuit(x + SHIFT, params)
          - 2.0 * circuit(x, params)
          + circuit(x - SHIFT, params))

def d2u_batch(xs, params):
    """Vectorised Laplacian at all x in xs."""
    return (circuit_batch(xs + SHIFT, params)
          - 2.0 * circuit_batch(xs, params)
          + circuit_batch(xs - SHIFT, params))

# ── Verification ───────────────────────────────────────────────────────────
p_test = np.random.default_rng(0).uniform(0, np.pi, 8)
x_v    = 0.4
h_ref  = 1e-7
fd1    = (circuit(x_v + h_ref, p_test) - circuit(x_v - h_ref, p_test)) / (2 * h_ref)
fd2    = (circuit(x_v + h_ref, p_test) - 2*circuit(x_v, p_test)
          + circuit(x_v - h_ref, p_test)) / h_ref**2
is1    = du_dx(x_v, p_test)
is2    = d2u_dx2(x_v, p_test)
print("Input-shift verification at x =", x_v)
print(f"  du/dx  : shift={is1:.8f}, FD(h=1e-7)={fd1:.8f}, diff={abs(is1-fd1):.2e}")
print(f"  d2u/dx2: shift={is2:.8f}, FD(h=1e-7)={fd2:.8f}, diff={abs(is2-fd2):.2e}")
print("Input-shift rule is EXACT for Ry(pi*x) encoding with SHIFT=pi/2.")


### 3.5  Spatial derivatives -- visualisation

In [ ]:
x_plot = np.linspace(0.05, 0.95, 80)
u_vals  = circuit_batch(x_plot, p_test)
du_vals = np.array([du_dx(x, p_test) for x in x_plot])
d2_vals = np.array([d2u_dx2(x, p_test) for x in x_plot])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_plot, u_vals,  lw=2.5, label=r"$u_\theta(x)$")
ax.plot(x_plot, du_vals, lw=2.0, ls='--', label=r"$u'_\theta(x)$  (input-shift)")
ax.plot(x_plot, d2_vals, lw=2.0, ls=':',  label=r"$u''_\theta(x)$  (input-shift)")
ax.set_xlabel('$x$')
ax.set_title("QNN output and exact spatial derivatives via input-shift rule  [SHIFT=pi/2]")
ax.legend()
plt.tight_layout(); plt.show()


### 3.6  Fourier analysis of the quantum ansatz

In [ ]:
# For Ry(pi*x) encoding, the output is periodic with period 2 in x.
# Accessible frequencies for n=2 qubits: omega in {0, 1/2, 1} (Schuld et al. 2021).
N_fft  = 2048
x_fft  = np.linspace(0, 2.0, N_fft, endpoint=False)
u_fft  = circuit_batch(x_fft, p_test)
freqs  = np.fft.rfftfreq(N_fft, d=2.0 / N_fft)
amps   = np.abs(np.fft.rfft(u_fft)) * 2 / N_fft

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_fft, u_fft, color=COLORS[0], lw=1.5)
axes[0].set_xlabel('$x$'); axes[0].set_ylabel(r'$u_\theta(x)$')
axes[0].set_title(r'QNN output over one period $[0,2]$  ($n=2$ qubits)')

max_show = 14
axes[1].stem(freqs[:max_show], amps[:max_show],
             linefmt=COLORS[0], markerfmt='o', basefmt='k-')
axes[1].set_xlabel(r'Frequency $\omega$'); axes[1].set_ylabel('Amplitude')
axes[1].set_title(r'Fourier spectrum: $n=2$ qubits, angle encoding  ($\omega_{\max}=1$)')
axes[1].set_xlim(-0.1, 4.0)
plt.tight_layout(); plt.show()

print("Significant frequency components (n=2 qubits):")
for i in np.argsort(amps[:14])[::-1][:5]:
    if amps[i] > 0.003:
        print(f"  omega = {freqs[i]:.3f},  amplitude = {amps[i]:.4f}")
print()
print("Theory (Schuld 2021): n=2 qubit angle encoding -> omega in {0, 1/2, 1}")


---
## 4  Classical Physics-Informed Neural Networks (week15_merged.tex, Section 4)

$$\mathcal{L}(\theta) = \frac{1}{N_r}\sum_k|\mathcal{N}[u_\theta](x_k)|^2 + \lambda\frac{1}{N_b}\sum_k|u_\theta(x_k^b)-g_k|^2$$

The NTK $\Theta_{kl} = \nabla_\theta f(x_k)^\top\nabla_\theta f(x_l)$ governs convergence per frequency.

**Failure modes motivating QPINNs:**

| Mode | Classical PINN | QPINN remedy |
|------|---------------|-------------|
| Spectral bias | Low-freq first | Explicit $\Omega$ |
| High-$d$ | Exponential cost | $O(d)$ qubits |
| Derivatives | Autograd (approximate) | Input-shift (exact) |
| Optimisation geometry | Euclidean | QFI / Fubini-Study |


---
## 5  QPINN Mathematical Foundation (week15_merged.tex, Sections 5-6)

### 5.1  Ansatz

$$u_\theta(x) = \langle 0|U^\dagger(x,\theta)\,Z_0\,U(x,\theta)|0\rangle, \quad U = W_L\cdots W_1\,S(x)$$

### 5.2  Unified shift framework

For **any** gate $e^{-i\phi P/2}$:
$$\frac{\partial}{\partial\phi}\langle O\rangle = \frac{1}{2}[\langle O\rangle_{\phi+\pi/2} - \langle O\rangle_{\phi-\pi/2}]$$

Two applications:
- $\phi = \theta_\mu$: **parameter-shift** — gradient for training
- $\phi = \pi x$ (since encoding is $R_y(\pi x)$): **input-shift** — spatial PDE derivative

**For linear PDE operator $\mathcal{D}$:**
$$\frac{\partial}{\partial\theta_\mu}\mathcal{D}[u_\theta] = \mathcal{D}\!\left[\frac{\partial u_\theta}{\partial\theta_\mu}\right]$$
so both shift rules commute with $\mathcal{D}$ — this makes QPINN training tractable.

### 5.3  QPINN loss

$$\mathcal{L}(\theta) = \frac{1}{N_r}\sum_k|\mathcal{D}[u_\theta](x_k)-f(x_k)|^2 + \lambda\frac{1}{N_b}\sum_k|u_\theta(x_k^b)-g(x_k^b)|^2$$

Total cost per gradient step: $O(N_p \times N_r)$ circuit evaluations.


---
## 6  The 1D Poisson Equation (week15_merged.tex, Section 7)

$$-\frac{d^2u}{dx^2} = f(x),\quad x\in(0,1),\quad u(0)=u(1)=0$$

**Exact solution** for $f(x)=\pi^2\sin(\pi x)$: $u^\star(x)=\sin(\pi x)$

### 6.1  Strong-form QPINN loss

$$\mathcal{L}_r = \frac{1}{N_r}\sum_k\left[-u''_\theta(x_k)-f(x_k)\right]^2$$

Second derivative via input-shift (SHIFT=pi/2):
$$u''_\theta(x) = u_\theta(x+\tfrac{\pi}{2}) - 2u_\theta(x) + u_\theta(x-\tfrac{\pi}{2}) \quad [3\text{ circuits}]$$

**Hard Dirichlet BCs** enforced exactly:
$\hat{u}(x) = u_\theta(x) - [(1-x)u_\theta(0) + x\,u_\theta(1)]$ — no penalty term needed.

### 6.2  Variational (Ritz) loss

$$\mathcal{L}_{\rm Ritz}(\theta) = \sum_k w_k\left[\tfrac{1}{2}(u'_\theta(x_k))^2 - f(x_k)\,\hat{u}(x_k)\right]$$

Only **first** derivatives needed (2 circuits per point); smoother landscape.
Equivalent to the **quantum Galerkin method** (quantum FEM).


### 6.4  Implementation and training

In [ ]:
def source(x):  return np.pi**2 * np.sin(np.pi * x)
def exact(x):   return np.sin(np.pi * x)

def loss_poisson(params, x_r):
    """Strong-form QPINN loss for -u'' = f(x) with hard Dirichlet BCs."""
    res = np.array([-d2u_dx2(x, params) - source(x) for x in x_r])
    return float(np.mean(res**2))

def grad_loss_poisson(params, x_r, shift=np.pi / 2):
    """Exact gradient via parameter-shift rule."""
    g = np.zeros(len(params))
    for mu in range(len(params)):
        p_plus  = params.copy(); p_plus[mu]  += shift
        p_minus = params.copy(); p_minus[mu] -= shift
        g[mu] = 0.5 * (loss_poisson(p_plus, x_r) - loss_poisson(p_minus, x_r))
    return g

np.random.seed(7)
N_r      = 20
x_r      = np.linspace(0.05, 0.95, N_r)
params   = np.random.uniform(0, np.pi, 8)
lr, b1, b2, eps_adam = 0.15, 0.9, 0.999, 1e-8
m_adam, v_adam = np.zeros(8), np.zeros(8)
history  = []
x_eval   = np.linspace(0, 1, 100)

print("Training QPINN:  -u'' = pi^2 sin(pi*x),  u(0)=u(1)=0")
print("Circuit: 2 qubits, Ry(pi*x) encoding, 2 var. layers (8 params)")
print("Derivatives: input-shift, SHIFT=pi/2  [exact -- no finite differences]")
print(f"{'Step':>6}  {'Loss':>12}  {'L2 error':>10}")
print("-" * 34)

for step in range(300):
    g = grad_loss_poisson(params, x_r)
    t = step + 1
    m_adam = b1 * m_adam + (1 - b1) * g
    v_adam = b2 * v_adam + (1 - b2) * g**2
    mc     = m_adam / (1 - b1**t)
    vc     = v_adam / (1 - b2**t)
    params -= lr * mc / (np.sqrt(vc) + eps_adam)
    lv     = loss_poisson(params, x_r)
    history.append(lv)
    if (step + 1) % 75 == 0:
        u_pred = np.array([circuit_dirichlet(x, params) for x in x_eval])
        l2     = np.sqrt(np.mean((u_pred - exact(x_eval))**2))
        print(f"{step+1:>6}  {lv:>12.6f}  {l2:>10.4f}")


### 6.5  Results

In [ ]:
x_plot_p = np.linspace(0, 1, 200)
u_qpinn  = np.array([circuit_dirichlet(x, params) for x in x_plot_p])
u_exact_ = exact(x_plot_p)
l2_error = np.sqrt(np.mean((u_qpinn - u_exact_)**2))

print(f"L2 error vs sin(pi*x): {l2_error:.4f}")
print(f"u_hat(0) = {circuit_dirichlet(0., params):.2e}  (hard BC -- exact 0)")
print(f"u_hat(1) = {circuit_dirichlet(1., params):.2e}  (hard BC -- exact 0)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(x_plot_p, u_exact_,  'k-',  lw=2.5, label=r'Exact $\sin(\pi x)$')
axes[0].plot(x_plot_p, u_qpinn,   'b--', lw=2,   label=f'QPINN (L2={l2_error:.3f})')
axes[0].scatter(x_r, np.zeros_like(x_r), s=25, c='r', zorder=5, label=f'{N_r} pts')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$u(x)$')
axes[0].set_title(r"Poisson: $-u''=\pi^2\sin(\pi x)$")
axes[0].legend(fontsize=9)

axes[1].semilogy(x_plot_p, np.abs(u_qpinn - u_exact_) + 1e-12, 'crimson', lw=1.5)
axes[1].set_xlabel('$x$'); axes[1].set_ylabel(r'$|u_\theta - u^\star|$')
axes[1].set_title('Pointwise error')

axes[2].semilogy(history, 'steelblue', lw=1.5)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Loss')
axes[2].set_title('Training loss (Adam + exact param-shift)')
plt.tight_layout(); plt.show()


### 6.6  Ritz formulation

In [ ]:
def loss_ritz(params, x_r):
    """
    Ritz (variational) QPINN: E[u] = sum_k [(1/2)(du_hat/dx)^2 - f*u_hat] / N_r.
    Needs only FIRST derivatives -- 2 circuit evaluations per point.
    Smoother loss landscape than strong form.
    """
    u_hat = np.array([circuit_dirichlet(x, params) for x in x_r])
    du    = np.array([du_dx(x, params) for x in x_r])
    return float(np.mean(0.5 * du**2 - source(x_r) * u_hat))

def grad_loss_ritz(params, x_r, shift=np.pi / 2):
    g = np.zeros(len(params))
    for mu in range(len(params)):
        p_plus  = params.copy(); p_plus[mu]  += shift
        p_minus = params.copy(); p_minus[mu] -= shift
        g[mu] = 0.5 * (loss_ritz(p_plus, x_r) - loss_ritz(p_minus, x_r))
    return g

np.random.seed(7)
params_ritz  = np.random.uniform(0, np.pi, 8)
m_r, v_r     = np.zeros(8), np.zeros(8)
hist_ritz    = []

for step in range(300):
    g   = grad_loss_ritz(params_ritz, x_r)
    t   = step + 1
    m_r = b1 * m_r + (1 - b1) * g
    v_r = b2 * v_r + (1 - b2) * g**2
    mc  = m_r / (1 - b1**t); vc = v_r / (1 - b2**t)
    params_ritz -= lr * mc / (np.sqrt(vc) + eps_adam)
    hist_ritz.append(loss_ritz(params_ritz, x_r))

x_plot_p = np.linspace(0, 1, 200)
u_ritz   = np.array([circuit_dirichlet(x, params_ritz) for x in x_plot_p])
l2_ritz  = np.sqrt(np.mean((u_ritz - exact(x_plot_p))**2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot_p, exact(x_plot_p), 'k-',  lw=2.5, label='Exact')
axes[0].plot(x_plot_p, u_qpinn,          'b--', lw=2,   label=f'Strong (L2={l2_error:.3f})')
axes[0].plot(x_plot_p, u_ritz,           'r-.',  lw=2,   label=f'Ritz (L2={l2_ritz:.3f})')
axes[0].set_xlabel('$x$'); axes[0].set_title('Strong form vs Ritz QPINN')
axes[0].legend(fontsize=9)

axes[1].semilogy(history,   'steelblue', lw=2, label='Strong form')
axes[1].semilogy(hist_ritz, 'crimson',   lw=2, label='Ritz')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss comparison'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"Ritz L2={l2_ritz:.4f}  |  Strong L2={l2_error:.4f}")


---
## 7  Harmonic Oscillator ODE (week15_merged.tex, Exercise 4)

$$\frac{d^2u}{dx^2} + \omega^2 u = 0, \qquad u(0)=0, \quad u'(0)=1$$

Exact solution: $u^\star(x) = \sin(\omega x)/\omega$

PDE residual: $r_k = u''_\theta(x_k) + \omega^2 u_\theta(x_k)$

Initial condition penalties (all via input-shift rule — exact):
$$\mathcal{L}_{\rm IC} = u_\theta(0)^2 + (u'_\theta(0) - 1)^2$$


In [ ]:
OMEGA = 2.0

def exact_harmonic(x):  return np.sin(OMEGA * x) / OMEGA

def loss_harmonic(params, x_r, lambda_ic=15.0):
    """
    QPINN loss for  u'' + omega^2*u = 0,  u(0)=0,  du_dx(0)=1.
    All derivatives via input-shift rule -- exact, no FD.
    """
    res  = np.array([d2u_dx2(x, params) + OMEGA**2 * circuit(x, params) for x in x_r])
    L_r  = float(np.mean(res**2))
    L_ic = circuit(0., params)**2 + (du_dx(0., params) - 1.0)**2
    return L_r + lambda_ic * L_ic

def grad_loss_harmonic(params, x_r, shift=np.pi / 2):
    g = np.zeros(len(params))
    for mu in range(len(params)):
        p_plus  = params.copy(); p_plus[mu]  += shift
        p_minus = params.copy(); p_minus[mu] -= shift
        g[mu] = 0.5 * (loss_harmonic(p_plus, x_r) - loss_harmonic(p_minus, x_r))
    return g

np.random.seed(3)
params_ho  = np.random.uniform(0, np.pi, 8)
x_r_ho     = np.linspace(0.02, 0.98, 20)
m_ho, v_ho = np.zeros(8), np.zeros(8)
hist_ho    = []

print(f"Harmonic oscillator: omega={OMEGA}, exact=sin({OMEGA}x)/{OMEGA}")
x_plot_p = np.linspace(0, 1, 200)
for step in range(500):
    g  = grad_loss_harmonic(params_ho, x_r_ho)
    t  = step + 1
    m_ho = b1 * m_ho + (1 - b1) * g
    v_ho = b2 * v_ho + (1 - b2) * g**2
    mc   = m_ho / (1 - b1**t); vc = v_ho / (1 - b2**t)
    params_ho -= 0.1 * mc / (np.sqrt(vc) + eps_adam)
    hist_ho.append(loss_harmonic(params_ho, x_r_ho))
    if (step + 1) % 125 == 0:
        u_ho  = circuit_batch(x_plot_p, params_ho)
        l2_ho = np.sqrt(np.mean((u_ho - exact_harmonic(x_plot_p))**2))
        print(f"  Step {step+1:4d}: loss={hist_ho[-1]:.5f},  L2={l2_ho:.4f}")

u_ho   = circuit_batch(x_plot_p, params_ho)
l2_ho  = np.sqrt(np.mean((u_ho - exact_harmonic(x_plot_p))**2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x_plot_p, exact_harmonic(x_plot_p), 'k-',  lw=2.5,
             label=fr'Exact $\sin({OMEGA}x)/{OMEGA:.0f}$')
axes[0].plot(x_plot_p, u_ho, 'b--', lw=2, label=f'QPINN (L2={l2_ho:.3f})')
axes[0].scatter(x_r_ho, np.zeros_like(x_r_ho), s=20, c='r', zorder=5)
axes[0].set_xlabel('$x$')
axes[0].set_title(fr'Harmonic oscillator: $\omega={OMEGA}$')
axes[0].legend()
axes[1].semilogy(hist_ho, 'steelblue', lw=1.5)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training loss')
plt.tight_layout(); plt.show()


---
## 8  Heat Equation: Space-Time Encoding (week15_merged.tex, Section 8)

$$\frac{\partial u}{\partial t} = \kappa\frac{\partial^2 u}{\partial x^2}, \quad x\in(0,1),\; t\in(0,T]$$
$$u(x,0)=\sin(\pi x), \quad u(0,t)=u(1,t)=0$$

Exact solution: $u^\star(x,t) = e^{-\kappa\pi^2 t}\sin(\pi x)$

**Space-time encoding** (week15_merged.tex Section 8):
$$S(x,t) = R_y(\pi x)\otimes R_y(\pi x)\otimes R_y(\pi t)$$
— 2 spatial qubits + 1 temporal qubit, all with angle encoding.

Derivatives via input-shift on the corresponding qubit registers:
$$\partial_t u \approx \tfrac{1}{2}[u(x,t+\tfrac{\pi}{2})-u(x,t-\tfrac{\pi}{2})], \qquad
\partial_{xx}u \approx u(x+\tfrac{\pi}{2},t)-2u(x,t)+u(x-\tfrac{\pi}{2},t)$$

Each PDE residual point: 5 circuit evaluations.


In [ ]:
KAPPA = 0.5
T_MAX = 0.3

# Three-qubit space-time circuit: qubits 0,1 = spatial, qubit 2 = temporal
Z_mat = np.diag([1., -1.]).astype(complex)
Z0_3q = np.kron(np.kron(Z_mat, I2), I2)   # Z x I x I

def var_layer_3q(p):
    """
    Three-qubit variational layer.
    Ry rotations on all qubits, then CNOT(0->1), then CNOT(1->2).
    6 parameters per layer.
    CNOT matrices use standard basis ordering |q0 q1 q2>.
    """
    Ry3 = np.kron(np.kron(ry(p[0]), ry(p[1])), ry(p[2]))
    # CNOT(0->1): flip q1 when q0=1
    # Basis order: |000>=0, |001>=1, |010>=2, |011>=3,
    #              |100>=4, |101>=5, |110>=6, |111>=7
    CNOT01 = np.array([
        [1,0,0,0, 0,0,0,0],
        [0,1,0,0, 0,0,0,0],
        [0,0,1,0, 0,0,0,0],
        [0,0,0,1, 0,0,0,0],
        [0,0,0,0, 0,0,1,0],
        [0,0,0,0, 0,0,0,1],
        [0,0,0,0, 1,0,0,0],
        [0,0,0,0, 0,1,0,0],
    ], dtype=complex)
    # CNOT(1->2): flip q2 when q1=1
    CNOT12 = np.array([
        [1,0,0,0, 0,0,0,0],
        [0,1,0,0, 0,0,0,0],
        [0,0,0,1, 0,0,0,0],
        [0,0,1,0, 0,0,0,0],
        [0,0,0,0, 1,0,0,0],
        [0,0,0,0, 0,1,0,0],
        [0,0,0,0, 0,0,0,1],
        [0,0,0,0, 0,0,1,0],
    ], dtype=complex)
    return CNOT12 @ CNOT01 @ Ry3

def circuit_xt(x, t, params):
    """
    Three-qubit QPINN for space-time (x, t).
    Encoding : Ry(pi*x) x Ry(pi*x) x Ry(pi*t)
    Ansatz   : 1 variational layer (6 parameters)
    Output   : <Z_0 x I x I>
    """
    state = np.zeros(8, dtype=complex); state[0] = 1.0
    enc   = np.kron(np.kron(ry(np.pi * x), ry(np.pi * x)), ry(np.pi * t))
    psi   = var_layer_3q(params) @ enc @ state
    return float((psi.conj() @ Z0_3q @ psi).real)

def du_dt_xt(x, t, params):
    """Temporal derivative via input-shift on t qubit."""
    return 0.5 * (circuit_xt(x, t + SHIFT, params)
                - circuit_xt(x, t - SHIFT, params))

def d2u_dx2_xt(x, t, params):
    """Spatial Laplacian via input-shift on x qubits."""
    return (circuit_xt(x + SHIFT, t, params)
          - 2.0 * circuit_xt(x, t, params)
          + circuit_xt(x - SHIFT, t, params))

def loss_heat(params, x_r, t_r, lambda_bc=8.0, lambda_ic=8.0):
    """QPINN loss for du/dt = kappa * d2u/dx2."""
    res  = np.array([
        du_dt_xt(x, t, params) - KAPPA * d2u_dx2_xt(x, t, params)
        for x, t in zip(x_r, t_r)
    ])
    L_r  = float(np.mean(res**2))
    t_bc = np.linspace(0.02, T_MAX, 8)
    L_bc = (np.mean([circuit_xt(0., t, params)**2 for t in t_bc])
           + np.mean([circuit_xt(1., t, params)**2 for t in t_bc]))
    x_ic = np.linspace(0.05, 0.95, 10)
    L_ic = float(np.mean([
        (circuit_xt(x, 0., params) - np.sin(np.pi * x))**2 for x in x_ic
    ]))
    return L_r + lambda_bc * L_bc + lambda_ic * L_ic

np.random.seed(5)
N_heat       = 25
x_r_heat     = np.random.uniform(0.05, 0.95, N_heat)
t_r_heat     = np.random.uniform(0.02, T_MAX, N_heat)
params_heat  = np.random.uniform(0, np.pi, 6)
m_ht, v_ht   = np.zeros(6), np.zeros(6)
hist_heat    = []

print("Training heat equation QPINN  (3 qubits: 2 spatial + 1 temporal)")
for step in range(600):
    g = np.zeros(6)
    for mu in range(6):
        pp = params_heat.copy(); pp[mu] += SHIFT
        pm = params_heat.copy(); pm[mu] -= SHIFT
        g[mu] = 0.5 * (loss_heat(pp, x_r_heat, t_r_heat)
                      - loss_heat(pm, x_r_heat, t_r_heat))
    t    = step + 1
    m_ht = b1 * m_ht + (1 - b1) * g
    v_ht = b2 * v_ht + (1 - b2) * g**2
    mc   = m_ht / (1 - b1**t); vc = v_ht / (1 - b2**t)
    params_heat -= 0.12 * mc / (np.sqrt(vc) + eps_adam)
    hist_heat.append(loss_heat(params_heat, x_r_heat, t_r_heat))
    if (step + 1) % 150 == 0:
        print(f"  Step {step+1:4d}: loss={hist_heat[-1]:.5f}")

x_vis    = np.linspace(0.02, 0.98, 60)
t_slices = [0.05, 0.15, 0.25]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t_s, col in zip(t_slices, COLORS[:3]):
    u_pred = np.array([circuit_xt(x, t_s, params_heat) for x in x_vis])
    u_true = np.exp(-KAPPA * np.pi**2 * t_s) * np.sin(np.pi * x_vis)
    l2_s   = np.sqrt(np.mean((u_pred - u_true)**2))
    axes[0].plot(x_vis, u_true,  '-',  c=col, lw=2.0, label=f't={t_s} exact')
    axes[0].plot(x_vis, u_pred,  '--', c=col, lw=1.5, label=f't={t_s} QPINN (L2={l2_s:.3f})')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$u(x,t)$')
axes[0].set_title(fr'Heat equation: $\kappa={KAPPA}$'); axes[0].legend(fontsize=8)
axes[1].semilogy(hist_heat, 'steelblue', lw=1.5)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Heat equation training loss')
plt.tight_layout(); plt.show()


---
## 9  Quantum Fisher Information and Natural Gradient (week15_merged.tex, Section 2)

$$F_{jk} = 4\,\mathrm{Re}\!\left(\langle\partial_j\psi|\partial_k\psi\rangle - \langle\partial_j\psi|\psi\rangle\langle\psi|\partial_k\psi\rangle\right)$$

**Exact diagonal entry** via parameter-shift:
$$F_{jj} = 1 - \left[\frac{u(\theta_j+\pi/2)+u(\theta_j-\pi/2)}{2}\right]^2$$

**Natural gradient** (week15_merged.tex Section 2):
$$\theta \leftarrow \theta - \eta\,(F+\varepsilon I)^{-1}\nabla_\theta\mathcal{L}$$

For QPINNs this is especially useful: PDE residual losses create highly non-isotropic
gradients, and QFI pre-conditioning aligns the update with the geometry of the quantum
state manifold (Fubini-Study metric).


In [ ]:
def compute_qfi_diagonal(params, x_sample, shift=np.pi / 2):
    """
    Exact diagonal QFI via (week15_merged.tex Section 2):
        F_jj = 4 * Var(P_j) = 1 - [(u(theta+pi/2) + u(theta-pi/2))/2]^2
    Averaged over collocation points x_sample.
    """
    n_p    = len(params)
    F_diag = np.zeros(n_p)
    for mu in range(n_p):
        p_plus  = params.copy(); p_plus[mu]  += shift
        p_minus = params.copy(); p_minus[mu] -= shift
        u_plus  = circuit_batch(x_sample, p_plus)
        u_minus = circuit_batch(x_sample, p_minus)
        F_diag[mu] = float(np.mean(1.0 - ((u_plus + u_minus) / 2.0)**2))
    return np.maximum(F_diag, 0.0)

def natural_gradient_step(params, grad, x_sample, eta=0.1, eps_reg=0.01):
    """Diagonal natural gradient: theta <- theta - eta * F_diag^{-1} * grad."""
    F_diag = compute_qfi_diagonal(params, x_sample)
    return params - eta * grad / (F_diag + eps_reg)

np.random.seed(7)
p_ng  = np.random.uniform(0, np.pi, 8)
p_ad  = p_ng.copy()
m_ng_, v_ng_ = np.zeros(8), np.zeros(8)
hist_ng, hist_ad2 = [], []
x_r_ng = np.linspace(0.05, 0.95, 15)
x_qfi  = np.linspace(0.1, 0.9, 8)

print("Comparing Adam vs diagonal-QFI natural gradient on Poisson...")
for step in range(200):
    g_ng = grad_loss_poisson(p_ng, x_r_ng)
    p_ng = natural_gradient_step(p_ng, g_ng, x_qfi, eta=0.08, eps_reg=0.02)
    hist_ng.append(loss_poisson(p_ng, x_r_ng))

    g_ad  = grad_loss_poisson(p_ad, x_r_ng)
    t     = step + 1
    m_ng_ = b1 * m_ng_ + (1 - b1) * g_ad
    v_ng_ = b2 * v_ng_ + (1 - b2) * g_ad**2
    mc_   = m_ng_ / (1 - b1**t); vc_ = v_ng_ / (1 - b2**t)
    p_ad -= 0.15 * mc_ / (np.sqrt(vc_) + eps_adam)
    hist_ad2.append(loss_poisson(p_ad, x_r_ng))

F_final = compute_qfi_diagonal(params, x_qfi)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(hist_ng,   'crimson',   lw=2, label='Natural gradient (diag. QFI)')
axes[0].semilogy(hist_ad2,  'steelblue', lw=2, label='Adam')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Adam vs. diagonal-QFI natural gradient on Poisson')
axes[0].legend()
axes[1].bar(np.arange(8), F_final, color=COLORS[0], alpha=0.8)
axes[1].set_xlabel(r'Parameter index $\mu$')
axes[1].set_ylabel(r'$F_{\mu\mu}$')
axes[1].set_title('Diagonal QFI at trained parameters')
plt.tight_layout(); plt.show()
print("QFI diagonal:", np.round(F_final, 3))


---
## 10  Expressivity: Fourier Analysis and Data Re-Uploading (week15_merged.tex, Section 8)

**Theorem** (Schuld et al. 2021): any QNN output with encoding gates $e^{-ixP/2}$ is:
$$u_\theta(x) = \sum_{\omega\in\Omega} c_\omega(\theta)\,e^{i\omega x}$$

**Data re-uploading** ($L$ layers, $n$ qubits): $\omega_{\max} = Ln/2$.

Design rule for PDEs: choose $n, L$ such that $Ln \geq 2\omega^\star$.

We verify this empirically by comparing the Fourier spectra of single-upload vs
re-uploading circuits, and by measuring how $\omega_{\max}$ grows with $L$.


In [ ]:
def circuit_reupload(x, params, L=2):
    """
    Data re-uploading QPINN (week15_merged.tex Section 1):
    U(x,theta) = prod_{l=1}^L [W_l(theta) * S(x)]
    Encoding: Ry(pi*x) x Ry(pi*x) at every layer.
    params: length 4*L.
    """
    state = np.zeros(4, dtype=complex); state[0] = 1.0
    U     = np.eye(4, dtype=complex)
    for l in range(L):
        enc = np.kron(ry(np.pi * x), ry(np.pi * x))   # Ry(pi*x) at every layer
        vl  = var_layer(params[4 * l : 4 * (l + 1)])
        U   = vl @ enc @ U
    psi = U @ state
    return float((psi.conj() @ Z0 @ psi).real)

N_fft2  = 2048
x_dense = np.linspace(0, 2.0, N_fft2, endpoint=False)
freqs2  = np.fft.rfftfreq(N_fft2, d=2.0 / N_fft2)

params_ru  = np.random.uniform(0, np.pi, 16)
vals_L1    = np.array([circuit_reupload(x, p_test[:4], L=1) for x in x_dense])
vals_L4    = np.array([circuit_reupload(x, params_ru,  L=4) for x in x_dense])
amp_L1     = np.abs(np.fft.rfft(vals_L1)) * 2 / N_fft2
amp_L4     = np.abs(np.fft.rfft(vals_L4)) * 2 / N_fft2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
max_show = 18
axes[0].stem(freqs2[:max_show], amp_L1[:max_show],
             linefmt=COLORS[0], markerfmt='o', basefmt='k-',
             label=r'Single upload $L=1$  ($\omega_{\max}=1$)')
axes[0].stem(freqs2[:max_show], amp_L4[:max_show],
             linefmt=COLORS[1], markerfmt='s', basefmt='k-',
             label=r'Re-uploading $L=4$  ($\omega_{\max}=4$)')
axes[0].set_xlabel(r'Frequency $\omega$'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('Fourier spectrum: single vs. re-uploading (n=2 qubits)')
axes[0].legend(fontsize=9)

L_vals = np.arange(1, 7)
max_freq_emp = []
for Lv in L_vals:
    p_l = np.random.uniform(0, np.pi, 4 * Lv)
    v_l = np.array([circuit_reupload(x, p_l, L=Lv) for x in x_dense])
    a_l = np.abs(np.fft.rfft(v_l)) * 2 / N_fft2
    nz  = np.where(a_l[:N_fft2 // 2] > 0.005)[0]
    max_freq_emp.append(freqs2[nz[-1]] if len(nz) else 0.0)

axes[1].plot(L_vals, L_vals * 1.0, 'k--', lw=2,
             label=r'Theory: $\omega_{\max}=L\cdot n/2=L$  ($n=2$)')
axes[1].plot(L_vals, max_freq_emp,  'ro-', ms=8, lw=2,
             label=r'Empirical $\omega_{\max}$')
axes[1].set_xlabel('Re-uploading layers $L$')
axes[1].set_ylabel(r'$\omega_{\max}$')
axes[1].set_title('Frequency grows with re-uploading depth')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
print("Empirical omega_max:", [f"{v:.1f}" for v in max_freq_emp])
print("Theory:    omega_max:", [f"{v:.1f}" for v in L_vals * 1.0])


---
## 11  Exercises (week15_merged.tex, Section 9)

### Exercise 1: Input-shift identity (analytical)

Prove that for $u_\theta(x)$ with encoding $S(x)=R_y(\pi x)^{\otimes n}$ and SHIFT=$\pi/2$:
$$\frac{d^2u_\theta}{dx^2} = u_\theta(x+\tfrac{\pi}{2}) - 2u_\theta(x) + u_\theta(x-\tfrac{\pi}{2})$$

*Hint:* Differentiate $\langle O\rangle_x = \langle 0|U^\dagger(x,\theta)\,O\,U(x,\theta)|0\rangle$
twice using $\partial_x R_y(\pi x) = -i\frac{\pi}{2}Y\,R_y(\pi x)$, then apply the
Euler identity to express the result as circuit evaluations at $x\pm\pi/2$.

### Exercise 2: Neumann boundary conditions


In [ ]:
def source_neumann(x):  return np.pi**2 * np.cos(np.pi * x)
def exact_neumann(x):   return np.cos(np.pi * x)

def loss_neumann(params, x_r, lambda_nbc=10.0):
    """
    QPINN with Neumann BCs: du/dx(0)=0, du/dx(1)=0.
    Neumann BCs computed via input-shift rule (exact).
    """
    res   = np.array([-d2u_dx2(x, params) - source_neumann(x) for x in x_r])
    L_r   = float(np.mean(res**2))
    L_nbc = du_dx(0., params)**2 + du_dx(1., params)**2
    return L_r + lambda_nbc * L_nbc

np.random.seed(11)
p_neu     = np.random.uniform(0, np.pi, 8)
m_n, v_n  = np.zeros(8), np.zeros(8)
hist_neu  = []

for step in range(400):
    g = np.zeros(8)
    for mu in range(8):
        pp = p_neu.copy(); pp[mu] += SHIFT
        pm = p_neu.copy(); pm[mu] -= SHIFT
        g[mu] = 0.5 * (loss_neumann(pp, x_r) - loss_neumann(pm, x_r))
    t   = step + 1
    m_n = b1 * m_n + (1 - b1) * g; v_n = b2 * v_n + (1 - b2) * g**2
    mc  = m_n / (1 - b1**t); vc = v_n / (1 - b2**t)
    p_neu -= 0.15 * mc / (np.sqrt(vc) + eps_adam)
    hist_neu.append(loss_neumann(p_neu, x_r))

x_plot_p  = np.linspace(0, 1, 200)
u_neu     = circuit_batch(x_plot_p, p_neu)
u_neu_ex  = exact_neumann(x_plot_p)
u_neu_c   = u_neu - np.mean(u_neu) + np.mean(u_neu_ex)
l2_neu    = np.sqrt(np.mean((u_neu_c - u_neu_ex)**2))

print(f"Neumann QPINN L2 error: {l2_neu:.4f}")
print(f"du/dx(0) via input-shift: {du_dx(0., p_neu):.6f}  (target: 0)")
print(f"du/dx(1) via input-shift: {du_dx(1., p_neu):.6f}  (target: 0)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_plot_p, u_neu_ex, 'k-',  lw=2.5, label=r'Exact $\cos(\pi x)$')
axes[0].plot(x_plot_p, u_neu_c,  'b--', lw=2,   label=f'QPINN Neumann (L2={l2_neu:.3f})')
axes[0].set_xlabel('$x$'); axes[0].legend()
axes[0].set_title(r"Neumann BCs: $-u''=\pi^2\cos(\pi x)$")
axes[1].semilogy(hist_neu, 'steelblue', lw=1.5)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_title('Neumann QPINN training loss')
plt.tight_layout(); plt.show()


### Exercise 3: Convergence with circuit depth

In [ ]:
def d2u_reupload(x, params, L):
    """Second derivative of re-uploading circuit via input-shift rule (scalar x)."""
    return (circuit_reupload(float(x) + SHIFT, params, L)
          - 2.0 * circuit_reupload(float(x), params, L)
          + circuit_reupload(float(x) - SHIFT, params, L))

def circuit_reupload_dirichlet(x, params, L):
    """Re-uploading circuit with hard Dirichlet BCs."""
    u0 = circuit_reupload(0., params, L)
    u1 = circuit_reupload(1., params, L)
    return circuit_reupload(x, params, L) - (1 - x) * u0 - x * u1

errors_L     = []
n_layer_list = [1, 2, 3, 4]
x_rL         = np.linspace(0.05, 0.95, 20)
x_plot_p     = np.linspace(0, 1, 200)
print(f"{'Layers':>8}  {'Params':>8}  {'L2 error':>10}")

for L in n_layer_list:
    np.random.seed(42)
    p_L  = np.random.uniform(0, np.pi, 4 * L)
    m_L  = np.zeros(4 * L); v_L = np.zeros(4 * L)

    for step in range(300):
        g_L = np.zeros(4 * L)
        for mu in range(4 * L):
            pp = p_L.copy(); pp[mu] += SHIFT
            pm = p_L.copy(); pm[mu] -= SHIFT
            rp = float(np.mean([(-d2u_reupload(x, pp, L) - source(x))**2 for x in x_rL]))
            rm = float(np.mean([(-d2u_reupload(x, pm, L) - source(x))**2 for x in x_rL]))
            bp = circuit_reupload(0., pp, L)**2 + circuit_reupload(1., pp, L)**2
            bm = circuit_reupload(0., pm, L)**2 + circuit_reupload(1., pm, L)**2
            g_L[mu] = 0.5 * ((rp + 10. * bp) - (rm + 10. * bm))
        t   = step + 1
        m_L = b1 * m_L + (1 - b1) * g_L
        v_L = b2 * v_L + (1 - b2) * g_L**2
        mc  = m_L / (1 - b1**t); vc = v_L / (1 - b2**t)
        p_L -= 0.15 * mc / (np.sqrt(vc) + eps_adam)

    u_L  = np.array([circuit_reupload_dirichlet(x, p_L, L) for x in x_plot_p])
    err  = np.sqrt(np.mean((u_L - exact(x_plot_p))**2))
    errors_L.append(err)
    print(f"{L:>8}  {4*L:>8}  {err:>10.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(n_layer_list, errors_L, 'ro-', ms=9, lw=2)
ax.set_xlabel('Variational layers $L$')
ax.set_ylabel('L2 error')
ax.set_title('QPINN accuracy vs circuit depth (re-uploading, Poisson)')
ax.set_xticks(n_layer_list)
plt.tight_layout(); plt.show()


---
## 12  Summary and Outlook

### Mathematical tools

| Tool | Formula | Cost |
|------|---------|------|
| Input-shift 1st | $\tfrac{1}{2}[u(x+\tfrac{\pi}{2})-u(x-\tfrac{\pi}{2})]$ | 2 circuits |
| Input-shift 2nd (Laplacian) | $u(x+\tfrac{\pi}{2})-2u(x)+u(x-\tfrac{\pi}{2})$ | 3 circuits |
| Parameter-shift | $\tfrac{1}{2}[f(\theta+\tfrac{\pi}{2})-f(\theta-\tfrac{\pi}{2})]$ | 2 circuits |
| Diagonal QFI | $1-[(u_++u_-)/2]^2$ | 2 circuits |
| Natural gradient | $\theta\leftarrow\theta-\eta(F+\varepsilon I)^{-1}\nabla\mathcal{L}$ | $O(N_p)$ |

### Problems solved

| Problem | Equation | BCs | Derivative method |
|---------|---------|-----|------------------|
| 1D Poisson | $-u''=\pi^2\sin(\pi x)$ | Dirichlet (hard) | Input-shift |
| Ritz Poisson | $-u''=\pi^2\sin(\pi x)$ | Dirichlet (hard) | Input-shift |
| Harmonic oscillator | $u''+\omega^2 u=0$ | ICs via input-shift | Input-shift |
| Heat equation | $\partial_t u=\kappa\partial_{xx}u$ | Dirichlet + IC | Space-time input-shift |
| Neumann BCs | $-u''=\pi^2\cos(\pi x)$ | Neumann via input-shift | Input-shift |

### References

1. M. Raissi et al., *Physics-informed neural networks*, J. Comput. Phys. **378**, 686 (2019).
2. A. Kyriienko et al., *Solving nonlinear DEs with differentiable quantum circuits*, PRA **103**, 052416 (2021).
3. M. Schuld et al., *Effect of data encoding on expressive power*, PRA **103**, 032430 (2021).
4. A. Perez-Salinas et al., *Data re-uploading for a universal quantum classifier*, Quantum **4**, 226 (2020).
5. J. Stokes et al., *Quantum natural gradient*, Quantum **4**, 269 (2020).
6. M. Cerezo et al., *Variational quantum algorithms*, Nat. Rev. Phys. **3**, 625 (2021).
